# Sparse direct solvers: orderings, fill-in, and the solve pipeline

A sparse direct solver factors `A` into triangular factors and then solves by
substitution. Elimination is not free in a sparse matrix. Eliminating one
unknown couples every pair of its neighbours, so the factors pick up nonzeros
where `A` had none. That **fill-in** decides how much memory the factorization
needs and how long it takes. The amount of fill depends almost entirely on the
**order** in which the unknowns are eliminated.

This notebook follows MATLAB's
[`colamd` example](https://www.mathworks.com/help/matlab/ref/colamd.html) with
mtl5, then goes further:

1. **The matrix.** Load `west0479` and look at its pattern.
2. **Orderings and fill.** Reorder it four ways, factor each, and look at the
   factors themselves.
3. **A symmetric problem.** Show that the winning ordering depends on the
   matrix's structure.
4. **The solve pipeline.** Separate analyze, factor, refactor and solve, and
   show what reusing the analysis saves.

It needs the `notebooks` extra (`pip install -e ".[notebooks]"`) for matplotlib
and SciPy. Every number printed here is computed when the notebook runs, and CI
runs it, so the `assert` cells catch an ordering that stops reducing fill.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

import mtl5
import mtl5.sparse as ms

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)


def spy(ax, M, title, color=SERIES[0], markersize=1.0):
    # MATLAB-style spy plot of an MTL5 or SciPy sparse matrix. precision="present"
    # draws every STORED entry, including ones whose value is exactly zero:
    # matplotlib's default hides those, but storage and work are spent on them.
    S = ms.to_scipy(M) if not sp.issparse(M) else M
    ax.spy(S, precision="present", markersize=markersize, color=color)
    ax.set_title(f"{title}\nnz = {S.nnz:,}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


def best_time(fn, repeats=5):
    # Best wall time of `repeats` calls, in milliseconds, and the last result.
    best, out = float("inf"), None
    for _ in range(repeats):
        t = time.perf_counter()
        out = fn()
        best = min(best, time.perf_counter() - t)
    return best * 1e3, out


print("mtl5", mtl5.__version__, "| orderings:", ms.orderings())

## 1. The matrix

`west0479` is a Harwell-Boeing test matrix of order 479. It models an
eight-stage Westerberg chemical distillation column, and it is unsymmetric.
The file in `data/` is the one Matrix Market distributes; `data/README.md`
gives the source.

The file stores 1910 entries, and 22 of them are explicit zeros. A stored zero
still occupies a slot in the factorization, so drop the zeros first.
MATLAB's `load west0479` reports 1887 nonzeros, which differs from the 1888
here by a single entry.

In [ ]:
A_file = mtl5.io.mm_read("data/west0479.mtx")
S = ms.to_scipy(A_file)
stored, explicit_zeros = S.nnz, int((S.data == 0).sum())
S.eliminate_zeros()
A = ms.from_scipy(S)
n = A.shape[0]

print(f"order {n}, {stored} stored entries, {explicit_zeros} explicit zeros -> nnz(A) = {A.nnz}")
assert A.nnz == stored - explicit_zeros == 1888

fig, ax = plt.subplots(figsize=(4, 4))
spy(ax, A, "west0479", markersize=2)
plt.show()

The eight column stages are visible as repeated blocks along the diagonal,
coupled by a few long-range entries.

## 2. Orderings and fill-in

`mtl5.sparse.splu` factors a column-permuted matrix with threshold partial
pivoting:

$$A[\text{row\_perm}][:,\ \text{col\_perm}] = L\,U$$

The **column** permutation comes from the fill-reducing ordering, chosen before
any arithmetic. The **row** permutation comes from pivoting, chosen during the
factorization for numerical stability. `mtl5.sparse.ordering(A, name)` returns
the column permutation on its own, so you can look at it before factoring.

| Ordering | What it minimizes | Natural fit |
|---|---|---|
| `natural` | nothing: the identity | a baseline |
| `colamd` | approximate minimum degree on the pattern of $A^TA$ | LU and QR of unsymmetric matrices: the fill of LU with partial pivoting is bounded by the Cholesky factor of $A^TA$ |
| `amd` | approximate minimum degree on the pattern of $A + A^T$ | Cholesky and $LDL^T$, and LU of matrices with a nearly symmetric pattern and a strong diagonal |
| `rcm` | bandwidth (reverse Cuthill-McKee) | banded or profile storage; not fill directly |

A permutation moves entries but never changes how many there are:

In [ ]:
ORDERINGS = ["natural", "colamd", "amd", "rcm"]
perms = {o: ms.ordering(A, o) for o in ORDERINGS}

fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for ax, o, c in zip(axes, ORDERINGS, SERIES):
    spy(ax, S[:, perms[o]], f"A[:, p]  ({o})", color=c)
plt.tight_layout()
plt.show()

assert all(S[:, p].nnz == A.nnz for p in perms.values())

The factors are where the orderings differ. Since #126 every factor object
exposes `L`, `U`, `row_perm` and `col_perm`, so the fill can be counted and
looked at, not just inferred from `.nnz`.

In [ ]:
b = S @ np.ones(n)
lus, rows = {}, []
for o in ORDERINGS:
    ms_factor, lu = best_time(lambda o=o: ms.splu(A, ordering=o))
    x = lu.solve(b).to_numpy()
    rows.append((o, lu.L.nnz, lu.U.nnz, lu.nnz, lu.nnz / A.nnz, ms_factor, np.abs(x - 1).max()))
    lus[o] = lu

header = ("ordering", "nnz(L)", "nnz(U)", "nnz(L+U)", "fill", "factor", "max|x-1|")
print("{:>8} {:>8} {:>8} {:>9} {:>6} {:>9} {:>9}".format(*header))
for o, nl, nu, nlu, ratio, t, err in rows:
    print(f"{o:>8} {nl:8,} {nu:8,} {nlu:9,} {ratio:5.1f}x {t:7.2f}ms {err:9.1e}")

# Fill regressions fail CI here. The solution check uses an absolute error: west0479
# is badly conditioned, so 1e-9 is what a backward-stable solve delivers on it.
assert lus["colamd"].nnz < lus["natural"].nnz / 2
assert lus["colamd"].nnz < min(lus["amd"].nnz, lus["rcm"].nnz)
assert all(err < 1e-7 for *_, err in rows)

Every ordering solves the system to the same accuracy. What they change is the
cost: `colamd` cuts the factors to about a third of the natural ordering's
fill. This is MATLAB's `spy(lu(A))` against `spy(lu(A(:,p)))`:

In [ ]:
def pattern(M):
    # The stored pattern with every value set to 1, so that adding two patterns
    # cannot cancel an entry or drop a stored zero.
    S = ms.to_scipy(M).tocsr()
    return sp.csr_matrix((np.ones(S.nnz), S.indices, S.indptr), shape=S.shape)


fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
for ax, o, c in zip(axes, ORDERINGS, SERIES):
    lu = lus[o]
    LU = pattern(lu.L) + pattern(lu.U)  # the diagonals overlap: L + U - I
    assert LU.nnz == lu.nnz - n
    spy(ax, LU, f"L + U - I  ({o})", color=c, markersize=0.6)
plt.tight_layout()
plt.show()

Some of those stored entries are exactly zero: the elimination cancelled
them. They are still fill. The solver allocated them and computed them, and
their positions are fixed by the pattern alone. The count and the plots
therefore include them:

In [ ]:
for o in ("natural", "colamd"):
    lu = lus[o]
    zeros = int((ms.to_scipy(lu.L).data == 0).sum() + (ms.to_scipy(lu.U).data == 0).sum())
    print(f"{o:>8}: {lu.nnz:6,} stored in L and U, {zeros:5,} of them exactly zero")

The factors also satisfy the identity exactly as documented. Reconstructing
`A` from them is a direct check that the permutations are the ones returned:

In [ ]:
lu = lus["colamd"]
L, U = ms.to_scipy(lu.L), ms.to_scipy(lu.U)
PAQ = S[lu.row_perm][:, lu.col_perm]
rel = spla.norm(L @ U - PAQ) / spla.norm(S)
print(f"||A[row_perm][:, col_perm] - L U|| / ||A|| = {rel:.1e}")
print(f"num_perturbed = {lu.num_perturbed}")
assert lu.num_perturbed == 0 and rel < 1e-14

### Comparing with MATLAB and SciPy

MATLAB's page reports 15,918 nonzeros for `lu(A)` and 5,920 for
`lu(A(:,p))`. Two things make the raw counts differ from the ones above:

- **Counting.** MATLAB's single-output `lu` returns `L + U - I`, so each
  diagonal entry is counted once. `L` here stores its unit diagonal, so
  subtract `n` to compare.
- **Pivoting.** MATLAB, SciPy (SuperLU) and mtl5 (Gilbert-Peierls LU) pick
  different pivots and break ties differently, so the factors differ. The
  natural ordering is the most sensitive, because without fill reduction every
  pivot choice spreads further.

What carries over between implementations is the size of the effect: column
minimum degree cuts the fill roughly threefold.

In [ ]:
matlab = {"natural": 15918, "colamd": 5920}
scipy_spec = {"natural": "NATURAL", "colamd": "COLAMD"}

print(f"{'ordering':>8} {'mtl5':>8} {'SciPy':>8} {'MATLAB':>8}    nnz(L+U-I)")
counts = {}
for o in ("natural", "colamd"):
    f = spla.splu(S.tocsc(), permc_spec=scipy_spec[o])
    counts[o] = (lus[o].nnz - n, f.L.nnz + f.U.nnz - n, matlab[o])
    print(f"{o:>8} {counts[o][0]:8,} {counts[o][1]:8,} {counts[o][2]:8,}")

for name, i in (("mtl5", 0), ("SciPy", 1), ("MATLAB", 2)):
    print(f"{name:>6}: colamd cuts fill {counts['natural'][i] / counts['colamd'][i]:.1f}x")

## 3. A symmetric problem: a different winner

`colamd` won above because `west0479` is unsymmetric. Now take the 2-D Poisson
matrix on an 80 × 80 grid: 6,400 unknowns, symmetric positive definite.

- Here $A + A^T$ is just the pattern of $A$, so `amd` orders the true
  elimination graph.
- `colamd` orders $A^TA$ instead. For a symmetric $A$ that is the pattern of
  $A^2$, which couples neighbours of neighbours. It is a denser graph than the
  one the factorization actually eliminates.
- Because the matrix is SPD, Cholesky applies too: it needs no pivoting and
  stores only `L`.

In [ ]:
P = mtl5.generators.poisson2d(80, 80)
SP = ms.to_scipy(P).tocsr()
N = P.shape[0]
print(f"poisson2d(80, 80): n = {N:,}, nnz(A) = {P.nnz:,}")

lu_p, chol_p = {}, {}
print(f"{'ordering':>8} {'LU nnz':>10} {'LU factor':>10} {'Cholesky nnz':>13} {'Chol factor':>12}")
for o in ORDERINGS:
    t_lu, lu_p[o] = best_time(lambda o=o: ms.splu(P, ordering=o), repeats=3)
    t_ch, chol_p[o] = best_time(lambda o=o: ms.cholesky(P, ordering=o), repeats=3)
    print(f"{o:>8} {lu_p[o].nnz:10,} {t_lu:8.1f}ms {chol_p[o].nnz:13,} {t_ch:10.1f}ms")

assert lu_p["amd"].nnz < lu_p["colamd"].nnz < lu_p["natural"].nnz
assert chol_p["amd"].nnz < chol_p["natural"].nnz / 3

The ranking has flipped: `amd` gives the least fill, and `colamd` is now
second. The Cholesky factors show why the orderings differ. They are drawn here
for a 24 × 24 grid, where the structure is visible at this size. `natural` keeps
the fill inside a band as wide as the grid, and fills the band completely.
`rcm` does the same inside a profile that narrows toward both ends. `amd` eliminates in an order that never lets
fill accumulate in one place:

In [ ]:
small = mtl5.generators.poisson2d(24, 24)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, o, c in zip(axes, ("natural", "rcm", "amd"), (SERIES[0], SERIES[3], SERIES[2])):
    L = ms.cholesky(small, ordering=o).L
    spy(ax, L, f"Cholesky L  ({o}, n = 576)", color=c, markersize=0.4)
plt.tight_layout()
plt.show()

That gives a rule of thumb, which the factorizations' default `ordering=`
arguments already follow:

| Matrix | Factorization | Ordering (default) |
|---|---|---|
| symmetric positive definite | `cholesky`, `supernodal_ldlt` | `amd` |
| symmetric indefinite | `ldlt`, `supernodal_ldlt` | `amd` |
| unsymmetric | `splu`, `supernodal_lu`, `qr` | `colamd` |
| unsymmetric, nearly symmetric pattern, strong diagonal | `splu` | try `amd` |
| a narrow band or profile is the goal | any | `rcm` |

When in doubt, measure: `ms.splu(A, ordering=o).nnz` for each `o` in
`ms.orderings()` costs a few factorizations and settles it.

## 4. The solve pipeline

A direct solve has four stages, and they have very different costs:

1. **Ordering.** A fill-reducing permutation, from the pattern alone.
2. **Symbolic analysis.** The structure of the factors (elimination tree,
   column counts), again from the pattern alone.
3. **Numeric factorization.** The arithmetic. For LU this includes pivoting.
4. **Solve.** Two triangular substitutions per right-hand side.

Constructing a factor object runs stages 1 to 3. `solve()` runs stage 4, and is
cheap enough to call for many right-hand sides. `refactor(A_new)` reruns
**only stage 3**, for a matrix with the same pattern. It reuses the ordering,
the symbolic structure and, for LU, the pivot sequence.

In [ ]:
lu = lu_p["amd"]
t_factor, _ = best_time(lambda: ms.splu(P, ordering="amd"), repeats=3)
t_refactor, _ = best_time(lambda: lu.refactor(P), repeats=3)
t_solve, _ = best_time(lambda: lu.solve(np.ones(N)), repeats=10)

print(f"factor   (stages 1-3)  {t_factor:7.2f} ms")
print(f"refactor (stage 3)     {t_refactor:7.2f} ms   {t_factor / t_refactor:4.1f}x cheaper")
print(f"solve    (stage 4)     {t_solve:7.2f} ms   {t_factor / t_solve:4.0f}x cheaper")

`refactor()` is built for problems whose values change while the pattern
stays fixed: implicit time stepping, Newton iterations, circuit transients,
parameter sweeps. Here twenty steps of $(A + c_k I)\,x = b$ are solved two ways.
In the first, each step is factored from scratch. In the second, one factor
object is refactored at every step.

In [ ]:
eye = sp.identity(N, format="csr")
systems = [(SP + 0.1 * k * eye).tocsr() for k in range(1, 21)]
assert all(
    np.array_equal(M.indptr, SP.indptr) and np.array_equal(M.indices, SP.indices) for M in systems
), "refactor() needs the same sparsity pattern at every step"
mats = [ms.from_scipy(M) for M in systems]
rhs = [M @ np.ones(N) for M in systems]

t = time.perf_counter()
for M, r in zip(mats, rhs):
    ms.splu(M, ordering="amd").solve(r)
t_fresh = time.perf_counter() - t

lu = ms.splu(mats[0], ordering="amd")
residuals = []
t = time.perf_counter()
for Ms, M, r in zip(systems, mats, rhs):
    lu.refactor(M)
    x = lu.solve(r).to_numpy()
    residuals.append(np.linalg.norm(Ms @ x - r) / np.linalg.norm(r))
t_reuse = time.perf_counter() - t

print(f"20 steps, factor each step: {t_fresh * 1e3:6.0f} ms")
print(f"20 steps, refactor:         {t_reuse * 1e3:6.0f} ms   ({t_fresh / t_reuse:.1f}x faster)")
print(f"worst relative residual:    {max(residuals):.1e}")
assert max(residuals) < 1e-12

The reused pivot sequence is also `refactor()`'s limit. The pivots were chosen
for the first matrix's values. If the values drift far enough that one of
those pivots becomes small, the refactored solve loses accuracy. `refactor()`
does not re-pivot for you. A cheap residual check, as above, is how you notice,
and a fresh `splu()` is the remedy.

## Summary

- Fill-in, not the nonzeros of `A`, decides the cost of a direct solve, and
  the elimination order decides the fill.
- Choose the ordering for the structure: `colamd` for unsymmetric LU and QR,
  `amd` for symmetric factorizations, `rcm` when a narrow band is the goal.
  The defaults follow this.
- The factors are inspectable. `L`, `U`, `row_perm` and `col_perm` satisfy
  `A[row_perm][:, col_perm] == L @ U`. For plots without matplotlib,
  `mtl5.io.spy(M, "file.png")` writes the pattern to a PNG.
- Analysis is a one-time cost when the pattern is fixed: use `refactor()`,
  and keep `solve()` for each new right-hand side.

Still to come in the sparse direct guide (#125): choosing a factorization, and
mixed-precision direct solves (a float32 factor refined with
`mtl5.mixed.iterative_refine`).